# Fixed Effects Modelling

In [1]:
import pandas as pd
import numpy as np

# Import data
arrivals_fx_df = pd.read_csv("../processed-csvs/combined_data.csv")
arrivals_fx_df.head()

,Unnamed: 0,Country/Continent,Arrivals,gdp_per_capita,gdp,inflation,unemployment,Under 1 week,1 and under 2 weeks,2 weeks and under 1 month,...,Education,Other & not stated,FLTS IN,PAX IN,SEATS IN,FLTS OUT,PAX OUT,SEATS OUT,Exchange Rate/TWI,Date
0,0,New Zealand,26140,12230.073455,4.274533e+10,2.602393,10.614,0.250648,0.270206,0.173036,...,0.055574,0.067072,480.0,79182.0,115531.0,480.0,69951.0,114745.0,1.3080,1991-01-01
1,1,New Zealand,26630,12230.073455,4.274533e+10,2.602393,10.614,0.228711,0.267852,0.202004,...,0.057144,0.061342,411.0,59457.0,99283.0,409.0,59593.0,98687.0,1.3105,1991-02-01
2,2,New Zealand,37290,12230.073455,4.274533e+10,2.602393,10.614,0.224430,0.305090,0.234579,...,0.019505,0.060842,472.0,69918.0,112703.0,467.0,66590.0,111108.0,1.3202,1991-03-01
3,3,New Zealand,32000,12230.073455,4.274533e+10,2.602393,10.614,0.285060,0.294345,0.184286,...,0.024939,0.044759,422.0,56856.0,98872.0,429.0,55901.0,100985.0,1.3326,1991-04-01
4,4,New Zealand,44500,12230.073455,4.274533e+10,2.602393,10.614,0.330855,0.292875,0.197460,...,0.017533,0.053466,447.0,62983.0,104890.0,445.0,58146.0,104503.0,1.3036,1991-05-01


In [3]:
# Clean Data
arrivals_fx_df = arrivals_fx_df.iloc[:,1:]

## Fixed Effects Model

Fixed effects model with statsmodels. Removing any NA's for now (or replacing with 0).

In [70]:
import statsmodels.formula.api as smf
from patsy import dmatrix

df = arrivals_fx_df.copy()
df.rename(columns={'Country/Continent':'Country'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])

# Use a consistent period for time FE (year-month)
df['ym'] = df['Date'].dt.to_period('M').astype(str)   # e.g. '2020-03'

# ------------- 2. Select & clean variables --------------
# Replace or drop rows with missing critical variables
df = df.dropna(subset=['Arrivals', 'Exchange Rate/TWI', 'gdp_per_capita', 'Date'])

# If any Arrivals = 0 (or Exchange = 0), add a small epsilon before log
eps = 1e-6
df['Arrivals_pos'] = df['Arrivals'].clip(lower=0) + eps
df['Exchange_pos'] = df['Exchange Rate/TWI'].clip(lower=0) + eps

# ------------- 3. Create log variables ------------------
df['ln_Arrivals'] = np.log(df['Arrivals_pos'])
df['ln_Exchange'] = np.log(df['Exchange_pos'])

# log controls (where appropriate)
df['ln_gdp_pc'] = np.log(df['gdp_per_capita'].clip(lower=eps))

# optionally log seats or pax in if you want supply control
df['ln_SEATS_IN'] = np.log(df['SEATS IN'].replace(0, np.nan).fillna(eps))
df['ln_PAX_IN'] = np.log(df['PAX IN'].replace(0, np.nan).fillna(eps))

# ------------- 4. (Optional) Additional control vars ----
# Keep non-logged controls as-is (inflation, unemployment)
# If these are proportions, scale appropriately.

# Add lags
df['Exchange_lag1'] = df.groupby('Country')['Exchange Rate/TWI'].shift(1)
df['Exchange_lag3'] = df.groupby('Country')['Exchange Rate/TWI'].shift(3)
df['Exchange_lag6'] = df.groupby('Country')['Exchange Rate/TWI'].shift(6)

df['Inflation_lag1'] = df.groupby('Country')['inflation'].shift(1)
df['Inflation_lag3'] = df.groupby('Country')['inflation'].shift(3)
df['Inflation_lag6'] = df.groupby('Country')['inflation'].shift(6)

df['Gdp_pc_lag1'] = df.groupby('Country')['gdp_per_capita'].shift(1)
df['Gdp_pc_lag3'] = df.groupby('Country')['gdp_per_capita'].shift(3)
df['Gdp_pc_lag6'] = df.groupby('Country')['gdp_per_capita'].shift(6)

# Log the lags
df['ln_Exchange_lag1'] = np.log(df['Exchange_lag1'].clip(lower=eps))
df['ln_Exchange_lag3'] = np.log(df['Exchange_lag3'].clip(lower=eps))
df['ln_Exchange_lag6'] = np.log(df['Exchange_lag6'].clip(lower=eps))

df['ln_Inflation_lag1'] = np.log(df['Inflation_lag1'].clip(lower=eps))
df['ln_Inflation_lag3'] = np.log(df['Inflation_lag3'].clip(lower=eps))
df['ln_Inflation_lag6'] = np.log(df['Inflation_lag6'].clip(lower=eps))

df['ln_Gdp_pc_lag1'] = np.log(df['Gdp_pc_lag1'].clip(lower=eps))
df['ln_Gdp_pc_lag3'] = np.log(df['Gdp_pc_lag3'].clip(lower=eps))
df['ln_Gdp_pc_lag6'] = np.log(df['Gdp_pc_lag6'].clip(lower=eps))

# Get rid of NA's
df = df[df["inflation"].isna() == False]
df = df[df["ln_Exchange_lag1"].isna() == False]
df = df[df["ln_Exchange_lag3"].isna() == False]
df = df[df["ln_Exchange_lag6"].isna() == False]

# ------------- 5. Build formula with country & month dummies --------
# Note: patsy / statsmodels will expand C(...) to dummies and drop one to avoid multicollinearity.
formula = "ln_Arrivals ~ ln_Exchange_lag1 + ln_gdp_pc + inflation + unemployment + ln_SEATS_IN + C(Country) + C(ym)"


# ------------- 6. Fit OLS with clustered standard errors (cluster by country) -----
model = smf.ols(formula=formula, data=df).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Country']}
)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            ln_Arrivals   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                -9.315e+11
Date:                Wed, 08 Oct 2025   Prob (F-statistic):               1.00
Time:                        13:48:48   Log-Likelihood:                -24735.
No. Observations:               16212   AIC:                         5.037e+04
Df Residuals:                   15764   BIC:                         5.381e+04
Df Model:                         447                                         
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

/Users/Justin/anaconda3/envs/data7001/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 447, but rank is 41
  warnings.warn('covariance of constraints does not have full '


### Fixed Effects Model with Linear Models

In [75]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

df['ym'] = df['Date'].dt.to_period('M')
#df['ym'] = df['ym'].to_timestamp()

# set multiindex
df_panel = df.set_index(['Country', 'Date'])

# select dependent & exog
y = df_panel['ln_Arrivals']
X = df_panel[
    [
        'ln_Exchange_lag1', 'ln_Exchange_lag3', 'ln_Exchange_lag6',
        'ln_Gdp_pc_lag1', 'ln_Gdp_pc_lag3', 'ln_Gdp_pc_lag6'
        ,'Inflation_lag1', 'Inflation_lag3', 'Inflation_lag6',
        'unemployment','ln_SEATS_IN']
]
X = sm.add_constant(X)

panel_model = PanelOLS(y, X, entity_effects=True, time_effects=True)
res = panel_model.fit(cov_type='clustered', cluster_entity=True)
print(res.summary)

/Users/Justin/anaconda3/envs/data7001/lib/python3.13/site-packages/linearmodels/panel/model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                          PanelOLS Estimation Summary                           
Dep. Variable:            ln_Arrivals   R-squared:                        0.0536
Estimator:                   PanelOLS   R-squared (Between):             -0.2318
No. Observations:               16188   R-squared (Within):               0.0123
Date:                Wed, Oct 08 2025   R-squared (Overall):             -0.0921
Time:                        13:54:03   Log-likelihood                 -2.47e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      81.052
Entities:                          42   P-value                           0.0000
Avg Obs:                       385.43   Distribution:                F(11,15734)
Min Obs:                       66.000                                           
Max Obs:                       402.00   F-statistic (robust):             12.792
                            

## Analysis of Fixed Effects Model

F-Statistic for statsmodel Fixed Effects model is 